In [0]:
%run /Shared/KrPrtnrs_service_principal

In [0]:
%run "./file_utils_nb"

In [0]:
# egress from azure to nifi 
# https://github.com/8451LLC/azure_egress_util
from datetime import datetime

wd = 'abfss://tmkrprtnrs-users@sa8451krprtnrdev.dfs.core.windows.net/l920466/ap-pos-feed'

In [0]:
from functools import reduce

for t in ['bask_store', 'dept_store']:

    files = [f[0] for f in dbutils.fs.ls(wd + "/send") if t in f[0]]

    dfs = [spark.read.csv(f, header=True, inferSchema=True) for f in files]

    # sanity check: flag schema drift instead of silently merging
    schemas = {tuple(d.columns) for d in dfs}
    if len(schemas) > 1:
        print(f"WARNING: schema mismatch across {t} files:")
        for s in schemas:
            print(s)

    df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

    filename = t + '_' + str(df.agg(f.min('fiscal_week')).collect()[0][0]) + \
        '_' + str(df.agg(f.max('fiscal_week')).collect()[0][0])

    save_csv(df, directory=wd + '/send', file_name=filename, append_timestamp=True)

    for file in files:
        dbutils.fs.rm(file, recurse=True)